# GĐ4 · Bước 2C — Kiểm kê đủ 7 nguồn và hỗ trợ lấy **bản nháp** từ CDC/NIDDK

**Bạn là người chạy từng ô Jupyter.** Notebook không huấn luyện, không tạo embeddings, không lập vector DB. Chỉ hai domain CDC/NIDDK có thể được hỗ trợ lấy nội dung, **và chỉ khi bạn đã xác minh quyền sử dụng và cập nhật registry**. WHO/Long Châu không có chế độ sao chép tự động.

Bản tải xuống chỉ vào thư mục `data/raw_staging/`; bạn phải kiểm tra thủ công trước khi chuyển vào `data/raw_approved/`.

## A. Kiểm kê trạng thái registry và các file `.txt`

In [3]:
from pathlib import Path
import csv
cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p/'registry'/'source_registry.csv').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Không tìm thấy registry/source_registry.csv trong thư mục hiện tại hoặc thư mục cha')
REGISTRY = ROOT/'registry'/'source_registry.csv'
RAW = ROOT/'data'/'raw_approved'
STAGING = ROOT/'data'/'raw_staging'
STAGING.mkdir(parents=True, exist_ok=True)
with REGISTRY.open(encoding='utf-8-sig', newline='') as f:
    sources = list(csv.DictReader(f))
assert len(sources) == 7 and len({r['source_id'] for r in sources}) == 7, 'Registry phải có đúng 7 ID khác nhau'
for r in sources:
    path = RAW / (r['source_id'] + '.txt')
    print(f"{r['source_id']:18} | {r['publisher']:10} | status={r['approval_status']:18} | exists={path.exists()} | bytes={path.stat().st_size if path.exists() else 0}")
print('TXT hiện có:', sum((RAW/(r['source_id']+'.txt')).exists() for r in sources),'/',len(sources))

CDC-T2D-001        | CDC        | status=approved_research  | exists=True | bytes=3349
CDC-RISK-002       | CDC        | status=approved_research  | exists=True | bytes=2891
NIDDK-DIAG-003     | NIDDK      | status=approved_research  | exists=True | bytes=0
NIDDK-T2D-004      | NIDDK      | status=approved_research  | exists=True | bytes=0
WHO-DIAB-005       | WHO        | status=approved_research  | exists=True | bytes=0
LC-HBA1C-006       | Long Chau  | status=approved_research  | exists=True | bytes=0
LC-STAGES-007      | Long Chau  | status=approved_research  | exists=True | bytes=0
TXT hiện có: 7 / 7


## B. Kiểm tra điều kiện trước khi tạo file nháp

**Không chạy bước tải nếu chưa phê duyệt thực sự**. Các điều kiện registry phải được chính bạn cập nhật dựa vào bằng chứng. Notebook không thay đổi bất kỳ trạng thái phê duyệt nào. Bạn có thể dùng riêng ô A để kiểm kê mà không chạy tải.

Chỉ chạy `pip` nếu thiếu `requests` hoặc `beautifulsoup4`:

```python
%pip install requests beautifulsoup4
```

In [4]:
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser
from datetime import datetime, timezone
import re, time
import requests
from bs4 import BeautifulSoup

APPROVED = {'approval_status':'approved_research', 'license_review_status':'verified', 'content_review_status':'owner_reviewed_research_only'}
ALLOWED_HOSTS = {'www.cdc.gov','cdc.gov','www.niddk.nih.gov','niddk.nih.gov'}

def candidate_ok(r):
    host = urlparse(r['canonical_url']).hostname
    return (
        r['publisher'] in {'CDC','NIDDK'} and host in ALLOWED_HOSTS
        and all(r.get(k,'').strip() == v for k,v in APPROVED.items())
        and bool(r.get('reviewer','').strip())
        and bool(r.get('source_retrieved_at_utc','').strip())
        and bool(r.get('license_evidence_url','').strip())
        and not (RAW/(r['source_id']+'.txt')).exists()
    )

candidates = [r for r in sources if candidate_ok(r)]
print('Nguồn CDC/NIDDK còn thiếu TXT và đã được duyệt thực tế:', [r['source_id'] for r in candidates])
print('WHO/Long Châu không được notebook tải toàn văn tự động.')

Nguồn CDC/NIDDK còn thiếu TXT và đã được duyệt thực tế: []
WHO/Long Châu không được notebook tải toàn văn tự động.


## C. Tải **bản nháp** (chỉ khi bạn đã kiểm tra quyền)

Lệnh dưới đây không tự chạy nếu không có nguồn đã duyệt. Nội dung HTML vẫn có thể chứa menu/footer hoặc văn bản không được phép. Tự rà soát và sửa trước khi dùng. Nếu website chặn robot hoặc request lỗi, **dừng và chuẩn bị bằng cách được trang cho phép**, không né chặn.

In [5]:
def allowed_by_robots(url, user_agent='*'):
    p = urlparse(url)
    robot_url = f'{p.scheme}://{p.netloc}/robots.txt'
    rp = RobotFileParser()
    rp.set_url(robot_url)
    try:
        rp.read()
        return rp.can_fetch(user_agent, url)
    except Exception as exc:
        print('Không xác minh được robots:', exc)
        return False

def extract_draft(html):
    soup = BeautifulSoup(html, 'html.parser')
    for tag in soup.select('script,style,nav,footer,header,aside,form,iframe,button,svg,noscript'):
        tag.decompose()
    main = soup.find('main') or soup.find('article') or soup
    out=[]
    for tag in main.find_all(['h1','h2','h3','p','li']):
        # tránh trùng text trong các thẻ cha
        if any(p.name in ('p','li') for p in tag.parents if p is not main):
            continue
        v = re.sub(r'\s+',' ',tag.get_text(' ',strip=True)).strip()
        if len(v) < 12:
            continue
        if tag.name in ('h1','h2','h3'):
            out.append('## '+v)
        else:
            out.append(v)
    return '\n\n'.join(out).strip()

session=requests.Session()
session.headers.update({'User-Agent': 'AcademicDocumentReview/0.1 (personal, low-volume)'} )
completed=[]
for r in candidates:
    url = r['canonical_url']
    sid = r['source_id']
    if not allowed_by_robots(url):
        print('SKIP (robots không cho phép hoặc không kiểm tra được):', sid)
        continue
    try:
        response = session.get(url, timeout=15, allow_redirects=True)
        response.raise_for_status()
        if urlparse(response.url).hostname not in ALLOWED_HOSTS:
            print('SKIP (redirect ngoài allowlist):', sid); continue
        draft=extract_draft(response.text)
        if len(draft.split()) < 25:
            print('SKIP (không trích xuất được nội dung hữu ích):',sid); continue
        target=STAGING/(sid+'.txt')
        if target.exists():
            print('SKIP (đã có bản nháp, không ghi đè):',sid); continue
        target.write_text(draft,encoding='utf-8')
        completed.append((sid,len(draft.split()),str(target)))
        print('BẢN NHÁP:', sid, 'words:',len(draft.split()), 'path:',target)
        time.sleep(2)
    except requests.RequestException as exc:
        print('SKIP (network):',sid,str(exc)[:200])
print('Đã tạo nháp (CHƯA DUYỆT):',completed)

Đã tạo nháp (CHƯA DUYỆT): []


## D. Phê duyệt thủ công và chạy lại Bước 2

1. Mở từng bản nháp trong `data/raw_staging/`, loại menu, quảng cáo, thẻ website, nội dung có bản quyền bên thứ ba, giữ tiêu đề phù hợp và đảm bảo nội dung gốc chính xác.
2. **Chỉ khi đủ quyền** hãy sao chép tệp đã rà soát vào `data/raw_approved/<source_id>.txt`. Giữ bản gốc cho các tệp đã tồn tại.
3. WHO và Long Châu: chỉ cập nhật permission/evidence sau khi nhận hoặc xác minh quyền cụ thể. Nếu không, giữ `reference_only`, không giả làm `approved_research`.
4. Chạy lại notebook `02_RAG_Document_Preparation.ipynb` → gửi bảng kết quả và `step2_preparation_report.json`.
5. Chúng ta sẽ **không chuyển Bước 3** cho đến khi thống nhất chính xác trạng thái của cả 7 nguồn.
